# Customer Churn Analysis — Reproducible Synthetic Demonstration

**Project type:** Independent portfolio exercise using simulated data.  
**Dataset source:** Generated in this notebook with NumPy; no real customer records or external dataset are used.  
**Purpose:** Demonstrate a repeatable analytics workflow: data audit, exploratory analysis, baseline comparison, a classification model, and interpretation.

> Important: This is a teaching simulation, not evidence about a real company, customer population, churn rate, or revenue loss. The simulated churn labels are generated from assumptions explicitly defined below.

## 1. Business question and analytical plan

**Question:** In a simulated subscription business, which customer characteristics are associated with a higher probability of churn, and how well can a simple model distinguish churn from retention?

Workflow:
1. Generate a reproducible synthetic customer table.
2. Audit shape, types, missing values, duplicates, and target balance.
3. Explore churn across selected customer attributes.
4. Split data before preprocessing to avoid leakage.
5. Compare a majority-class baseline with logistic regression.
6. Interpret model coefficients cautiously and document limitations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n_customers = 3000

# All records and churn labels below are simulated, not observed customers.
tenure_months = rng.integers(1, 73, size=n_customers)
monthly_charges = np.clip(rng.normal(72, 18, size=n_customers), 20, 130).round(2)
contract = rng.choice(['Month-to-month', 'One year', 'Two year'], size=n_customers, p=[0.55, 0.27, 0.18])
internet_service = rng.choice(['Fiber', 'DSL', 'None'], size=n_customers, p=[0.48, 0.42, 0.10])
support_tickets_90d = rng.poisson(1.1, size=n_customers)
autopay = rng.choice(['Yes', 'No'], size=n_customers, p=[0.62, 0.38])

# The following coefficients deliberately encode simulated relationships.
log_odds = (
    -1.15
    + 0.030 * (monthly_charges - 70)
    - 0.035 * (tenure_months - 24)
    + 1.05 * (contract == 'Month-to-month')
    - 0.55 * (contract == 'Two year')
    + 0.38 * (internet_service == 'Fiber')
    + 0.42 * support_tickets_90d
    + 0.65 * (autopay == 'No')
)
churn_probability = 1 / (1 + np.exp(-log_odds))
churn = rng.binomial(1, churn_probability)

df = pd.DataFrame({
    'tenure_months': tenure_months,
    'monthly_charges': monthly_charges,
    'contract': contract,
    'internet_service': internet_service,
    'support_tickets_90d': support_tickets_90d,
    'autopay': autopay,
    'churn': churn
})
df.head()

## 2. Data audit

Check the dataset structure and basic quality before analysis. Since the data is generated in code, the audit should still be performed rather than assumed.

In [ ]:
print('Rows, columns:', df.shape)
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget counts:\n', df['churn'].value_counts().rename(index={0:'Retained', 1:'Churned'}))
print('\nTarget rate: {:.1%}'.format(df['churn'].mean()))
df.describe(include='all').T

## 3. Exploratory analysis

These charts describe patterns in this simulated sample only. They should not be generalized to real customers.

In [ ]:
churn_by_contract = df.groupby('contract', observed=True)['churn'].mean().sort_values(ascending=False)
print('Simulated churn rate by contract:\n', churn_by_contract.map('{:.1%}'.format))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_by_contract.plot(kind='bar', ax=axes[0], title='Simulated churn rate by contract')
axes[0].set_ylabel('Churn rate')
axes[0].tick_params(axis='x', rotation=20)
df.boxplot(column='tenure_months', by='churn', ax=axes[1])
axes[1].set_title('Tenure by simulated churn label')
axes[1].set_xlabel('Churn (0 = retained, 1 = churned)')
axes[1].set_ylabel('Months')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
ticket_churn = df.groupby('support_tickets_90d', observed=True)['churn'].agg(['count', 'mean'])
ticket_churn = ticket_churn.rename(columns={'mean':'simulated_churn_rate'})
display(ticket_churn.style.format({'simulated_churn_rate':'{:.1%}'}))

fig, ax = plt.subplots(figsize=(7, 4))
ticket_churn['simulated_churn_rate'].plot(kind='bar', ax=ax, title='Simulated churn rate by support tickets')
ax.set_xlabel('Support tickets in prior 90 days')
ax.set_ylabel('Simulated churn rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## 4. Train/test split and modelling

A stratified holdout split is created before fitting preprocessing. The majority-class baseline is included so model performance is not judged in isolation.

In [ ]:
X = df.drop(columns='churn')
y = df['churn']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = ['tenure_months', 'monthly_charges', 'support_tickets_90d']
categorical_features = ['contract', 'internet_service', 'autopay']
preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
model = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

baseline_pred = np.zeros(len(y_test), dtype=int)
results = pd.DataFrame([
    {'model':'Majority-class baseline',
     'accuracy':accuracy_score(y_test, baseline_pred),
     'precision':precision_score(y_test, baseline_pred, zero_division=0),
     'recall':recall_score(y_test, baseline_pred, zero_division=0),
     'f1':f1_score(y_test, baseline_pred, zero_division=0)},
    {'model':'Logistic regression',
     'accuracy':accuracy_score(y_test, pred),
     'precision':precision_score(y_test, pred, zero_division=0),
     'recall':recall_score(y_test, pred, zero_division=0),
     'f1':f1_score(y_test, pred, zero_division=0)}
])
results.round(3)

In [ ]:
print('ROC-AUC: {:.3f}'.format(roc_auc_score(y_test, prob)))
print('\nClassification report:\n', classification_report(y_test, pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Retained', 'Churned'])
plt.title('Logistic regression — test set')
plt.tight_layout()
plt.show()

## 5. Model interpretation

Logistic regression coefficients show the direction of association in this fitted model, conditional on the encoded features. They do **not** establish causation. Because the data generator intentionally encoded relationships, this section checks whether the model can recover some of those designed patterns.

In [ ]:
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
coef_table = pd.DataFrame({'feature':feature_names, 'coefficient':coefficients})
coef_table['absolute_coefficient'] = coef_table['coefficient'].abs()
coef_table.sort_values('absolute_coefficient', ascending=False).drop(columns='absolute_coefficient').head(15)

## 6. Findings, recommendations, and limitations

### What this exercise demonstrates
- A reproducible synthetic dataset can be generated with a fixed random seed.
- Data quality checks can be automated and documented.
- Grouped churn rates help explore relationships before modelling.
- A baseline provides context for classification metrics.
- A preprocessing/model pipeline keeps transformations inside the training workflow and reduces leakage risk.

### Example business actions to test with real data
1. Review the customer journey for month-to-month subscribers before renewal.
2. Investigate recurring support issues and measure whether resolution quality predicts retention.
3. Compare retention outcomes for customers using automatic payments, while checking for confounding factors.
4. Evaluate model thresholds using the relative costs of missed churners and unnecessary retention outreach.

### Limitations
- All records and labels are simulated; there is no external dataset source.
- Churn probability is generated from assumptions written in the notebook, so model performance is not evidence of real-world predictive power.
- No causal conclusions can be drawn.
- The exercise does not estimate real revenue loss, customer lifetime value, or company performance.
- A real deployment would require a documented data source, privacy review, temporal validation, calibration, monitoring, and stakeholder review.

### Next step for a real-world case study
Replace this simulation with a documented public or authorized dataset. Repeat the audit and analysis, cite the source and license, and re-run the full workflow before making claims about actual customer behaviour.